# Step 7: Prepare and Launch Dashboards


This notebook prepares the files used by the interactive Spatial-VTK dashboards. For small tutorial data it can build dashboard datasets in the notebook. For full runs, keep dashboard preparation as a CLI step so the notebook does not load multi-GB metric or QC tables.

## Imports

These helpers write dashboard datasets and make readable dashboard table previews.


In [1]:
from spatial_vtk.config.notebook import notebook_timer, register_svtk_cell_timer

with notebook_timer():
    from pathlib import Path

    import pandas as pd

    from spatial_vtk.config import SpatialVTKConfig
    from spatial_vtk.config.outputs import resolve_output_path
    from spatial_vtk.io import load_output_table
    from spatial_vtk.visualize.dashboard import (
        display_table,
        write_dashboard_metric_dataset,
        write_dashboard_summary_dataset,
    )
    register_svtk_cell_timer()

Run time: 1.16 s


## Configuration

Load the config and choose dashboard output folders and ports.


In [2]:
from pathlib import Path

# Use the repository root so paths match the public source checkout.
repo_root = Path.cwd()
config_path = repo_root / "data/examples/configuration/example_spatial_vtk_config.yaml"

# Load the tutorial run scenario and make it the active config for later package calls.
cfg = SpatialVTKConfig.from_file(config_path, run_scenario="tutorial").activate()

notebook_overrides = {
    "metrics_port": 8501,
    "qc_port": 8502,
    # Keep this True for tutorial-sized data. Set False for full runs and use the printed CLI commands.
    "prepare_dashboards_locally": True,
    "preview_rows": 5,
}

Run time: 20.9 ms


## Prepare Dashboard Datasets

Large metric and QC tables should stay on disk. This cell resolves the standard output paths, optionally prepares dashboard metric/summary datasets for small tutorial data, and prints the equivalent CLI command for full runs.

In [3]:
metrics_path = resolve_output_path("metrics_enriched", kind="table")
qc_inventory_path = resolve_output_path("qc_inventory", kind="table")
metrics_dashboard_root = resolve_output_path("metrics_dashboard", kind="dashboard", create_parent=True)
dashboard_summary_root = resolve_output_path("dashboard_summaries", kind="dashboard", create_parent=True)

prepare_dashboards_locally = bool(notebook_overrides.get("prepare_dashboards_locally", True))

if prepare_dashboards_locally:
    # Local preparation is intended for tutorial-sized data. Full runs should use the CLI command below.
    write_dashboard_metric_dataset(metrics_path, metrics_dashboard_root, partitioned=True)
    write_dashboard_summary_dataset(metrics_dashboard_root, dashboard_summary_root, format="parquet")
else:
    print("Skipping local dashboard preparation. Run this after metric outputs are available:")

metrics_outputs_command = " ".join(
    [
        "svtk metrics outputs",
        f"--metrics {metrics_path}",
        f"--output-dir {Path(metrics_path).parent}",
        "--format parquet",
        "--dashboard-partitioned",
    ]
)
print(metrics_outputs_command)
print(f"metrics dashboard root: {metrics_dashboard_root}")
print(f"summary root: {dashboard_summary_root}")
print(f"QC inventory: {qc_inventory_path}")

,event_id,station,network,component,model,band,metric,metric_group,period_s,value_obs,...,strike,dip,rake,usgs_url,observed_pickle,event_json,synthetic_mseed,selected_station_count,overlapping_broadband_station_count,event_count
0,ci38038071,BFS,CI,Z,cvmsi,1-2 sec,PGA,amplitude,NaN,0.118,...,312.0,79.0,178.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/synth...,30,52,4.0
1,ci38038071,BHP,CI,Z,cvmsi,1-2 sec,PGV,amplitude,NaN,5.410,...,312.0,79.0,178.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/synth...,30,52,5.0
2,ci38695658,BLC,CI,R,cvmsi,2-3 sec,PSA,spectral,2.0,0.284,...,96.0,48.0,108.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/synth...,30,75,4.0
3,ci39812319,BRE,CI,T,cvmsi,3-5 sec,FAS,spectral,3.0,0.037,...,122.0,85.0,179.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/synth...,30,77,5.0
4,ci38695658,CAC,CI,Z,cvmsi,1-2 sec,PGA,amplitude,NaN,0.096,...,96.0,48.0,108.0,https://earthquake.usgs.gov/earthquakes/eventp...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/obser...,data/examples/example_five_event_subset/synth...,30,75,NaN


Run time: 41.40 s


## Preview the Metrics Table

Preview only a few rows. Do not load full dashboard tables in the notebook for large runs.

In [4]:
# Show a dashboard-style preview with readable metric names, period bands, and column headers.
preview_columns = [
    "event_id",
    "station",
    "component",
    "band",
    "metric",
    "value_obs",
    "value_syn",
    "log2_residual",
    "anderson_2004_gof",
]

if metrics_path.suffix.lower() in {".parquet", ".pq"}:
    metrics_preview = pd.read_parquet(metrics_path, columns=[column for column in preview_columns if column], engine="auto").head(notebook_overrides["preview_rows"])
else:
    metrics_preview = pd.read_csv(metrics_path, usecols=lambda column: column in set(preview_columns), nrows=notebook_overrides["preview_rows"], low_memory=False)

display_table(
    metrics_preview,
    columns=[column for column in preview_columns if column in metrics_preview.columns],
    max_rows=notebook_overrides["preview_rows"],
)

,Event ID,Station,Component,Period Band,Metric,Observed Value,Synthetic Value,Log2 Residual,Anderson 2004 GOF
0,ci38038071,BFS,Z,1-2 sec,Peak acceleration (PGA),0.118,0.104,0.184,8.7
1,ci38038071,BHP,Z,1-2 sec,Peak velocity (PGV),5.410,6.220,-0.201,7.9
2,ci38695658,BLC,R,2-3 sec,Pseudo-spectral acceleration (PSA),0.284,0.211,0.429,8.2
3,ci39812319,BRE,T,3-5 sec,Fourier amplitude spectrum (FAS),0.037,0.052,-0.491,7.4
4,ci38695658,CAC,Z,1-2 sec,Peak acceleration (PGA),0.096,0.089,0.109,8.9


Run time: 10.5 ms


## Build Launch Commands

Run these commands in a terminal when you want to open the interactive dashboards.


In [5]:
# Build portable CLI commands for launching the local Streamlit dashboards.
metrics_command = " ".join(
    [
        "svtk dashboard metrics",
        f"--metrics-root {metrics_dashboard_root}",
        f"--summary-root {dashboard_summary_root}",
        f"--port {notebook_overrides['metrics_port']}",
    ]
)
qc_command = " ".join(
    [
        "svtk dashboard qc",
        f"--trace-summary {qc_inventory_path}",
        f"--port {notebook_overrides['qc_port']}",
    ]
)

print("Metrics dashboard:")
print(metrics_command)
print()
print("QC dashboard:")
print(qc_command)

Metrics dashboard:
svtk dashboard metrics --port 8501

QC dashboard:
svtk dashboard qc --port 8502
Run time: 1.0 ms
